In [1]:
# ¿Cómo preparamos los abstracts de Scopus para identificar sus términos distintivos sin que textos vacíos, fórmulas retóricas y avisos de copyright distorsionen la representación?

import pandas as pd

dataframe = pd.read_csv("../data/scopus_abstracts.csv.gz", compression="gzip")
dataframe[["scopus_record_id", "publication_title", "abstract"]].head()

,scopus_record_id,publication_title,abstract
0,2-s2.0-105042759102,Strategic Pathways in Malaysia’s Fintech Secto...,This study examines the relationship between e...
1,2-s2.0-85175999008,"ELI Principles on Blockchain Technology, Smart...",The project started in 2018 following a Use-Ca...
2,2-s2.0-85149119289,Bitland-A Decentralized Commercial Real Estate...,Real estate has always been a significant inve...
3,2-s2.0-84931062324,"The politics of person, property, technology: ...",Global policy organizations have emphasized ca...
4,2-s2.0-85049126196,Planned preventative maintenance and proptech,[No abstract available]


In [2]:
# Los abstracts muy breves suelen ser marcadores de ausencia y no aportan contenido para una representación documento–término.

dataframe["abstract_word_count"] = dataframe["abstract"].str.split().str.len()
dataframe.sort_values("abstract_word_count")[["abstract_word_count", "abstract"]].head(
    20
)

,abstract_word_count,abstract
203,3,[No abstract available]
375,3,[No abstract available]
39,3,[No abstract available]
188,3,[No abstract available]
342,3,[No abstract available]
106,3,[No abstract available]
219,3,[No abstract available]
175,3,[No abstract available]
25,3,[No abstract available]
328,3,[No abstract available]


In [3]:
# Se excluyen marcadores conocidos de ausencia y textos demasiado cortos antes de transformar sus palabras.

minimum_abstract_words = 20
missing_abstract_markers = {"[no abstract available]"}

dataframe["has_missing_abstract_marker"] = (
    dataframe["abstract"].str.casefold().isin(missing_abstract_markers)
)
dataframe["has_sufficient_content"] = ~dataframe[
    "has_missing_abstract_marker"
] & dataframe["abstract_word_count"].ge(minimum_abstract_words)
excluded_abstracts = int((~dataframe["has_sufficient_content"]).sum())
dataframe = dataframe.loc[dataframe["has_sufficient_content"]].copy()
dataframe["normalized_text"] = dataframe["abstract"].str.lower()

excluded_abstracts, len(dataframe)

(20, 385)

In [4]:
# Los finales alineados a la derecha y ordenados de derecha a izquierda agrupan avisos editoriales similares.


def print_aligned_text_tails(texts, tail_word_count=15, tail_sample_size=20):
    evidence = pd.DataFrame(
        {
            "text_tail": texts.str.split().str[-tail_word_count:].str.join(" "),
            "reversed_text": texts.str[::-1],
        }
    ).sort_values("reversed_text")
    text_tails = evidence["text_tail"].head(tail_sample_size)
    tail_width = text_tails.str.len().max()

    for text_tail in text_tails:
        print(text_tail.rjust(tail_width))


print_aligned_text_tails(dataframe["normalized_text"])

                                          has been considered the last human frontier in the technological world. © 2023 the author(s)
                                       that can be used alongside the classic world city connectivity indicators. © 2023 the author(s)
                                     the nigerian environment. the implication of key findings is also discussed. © 2024 the author(s)
                                 igi global scientific publishing. all rights reserved. no part of this publication may be reproduced,
                                             new figure displaccer, paet, citytim = proptech platforms and smart city systems © 2026 .
                              with gm foods and recombinant technology. © 2012 association of food scientists & technologists (india).
                          as it has not been much explained. © 2019 international business information management association (ibima).
the original work is attributed as specified on the sag

In [5]:
# Los avisos editoriales al final no describen el contenido del artículo y añadirían términos irrelevantes al vocabulario.

copyright_pattern = r"\s*(?:(?:copyright\s*)?©|copyright\b|all rights reserved\b).*?$"
dataframe["has_copyright_notice"] = dataframe["normalized_text"].str.contains(
    copyright_pattern,
    case=False,
    regex=True,
)
dataframe["has_copyright_notice"].sum()

335

In [6]:
# Se elimina el aviso editorial antes de conservar una versión del abstract orientada al contenido.

dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
    copyright_pattern,
    "",
    case=False,
    regex=True,
)

In [7]:
# La misma evidencia permite verificar que el reemplazo retiró los finales editoriales repetidos.

print_aligned_text_tails(dataframe["normalized_text"])

              obtained, the proposed methodology tends to be efficient in terms of privacy protection, receiver anonymity,
                                                when x was 5% and it had the highest remanent polarization when x was 15%.
                academics, property students and senior school executives (e.g. deputy dean and deputy head of education).
                         a lot in the transformation of traditional real estate firms into digitally native brands (dnbs).
bim-gis semi-automatic prototype based on the depreciated reconstruction cost through an experimentation in rende (italy).
            also included in the business & management collection which can be accessed at https://hstalks.com/ business/.
          is also included in the business & management collection which can be accessed at https://hstalks.com/business/.
          is also included in the business & management collection which can be accessed at https://hstalks.com/business/.
                

In [8]:
# Se presentan conectores discursivos de varias palabras que aparecen en el corpus; son expresiones generales del inglés y no fórmulas retóricas de abstracts.

selected_connectors = [
    "as well as",
    "due to",
    "in terms of",
    "in order to",
    "according to",
    "as a result",
    "in the process",
    "in the future",
    "in relation to",
    "in that",
    "as such",
    "in response to",
    "leading to",
    "leads to",
    "at least",
    "based on",
    "used in",
    "and its",
    "some of",
    "to this end",
]

connector_counts = pd.DataFrame(
    {
        "connector": selected_connectors,
        "occurrences": [
            dataframe["normalized_text"].str.count(r"\b" + connector + r"\b").sum()
            for connector in selected_connectors
        ],
    }
).query("occurrences > 0")

connector_counts.sort_values("occurrences", ascending=False)

,connector,occurrences
15,based on,96
1,due to,53
0,as well as,43
17,and its,27
3,in order to,24
2,in terms of,23
16,used in,21
4,according to,20
18,some of,14
5,as a result,12


In [9]:
# Se marcan los conectores sin eliminarlos todavía para que la etapa posterior pueda decidir si conservan o no valor analítico.

for connector in sorted(selected_connectors, key=len, reverse=True):
    dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
        r"\b" + connector + r"\b",
        connector.replace(" ", "_"),
        regex=True,
    )

abstract_previews = dataframe["normalized_text"].str[:130]
marked_abstracts = abstract_previews[abstract_previews.str.contains("_")].head(8)

for abstract in marked_abstracts:
    print(abstract)

lychee (litchi chinensis) is one of the economic fruit in thailand with high nutritional value and health benefits. lychee and_its
the structural and optoelectronic properties, as_well_as epitaxial growth peculiarities of thin film strained layer si1-xgex/si he
this paper adopts method of theory and practice to study the digital real estate management system items. based_on theoretical res
due_to their biocompatibility and bioactivity, bioactive glasses are used as highly effective implant materials throughout the hum
this article deals with ongoing attempts to recover heat and greywater at property level, based_on an in-depth study of stockholm,
digital marketing in real estate companies is a prominent trend due_to superior network infrastructure and rising smartphone usage
property and construction are some_of world's largest industries, but also some_of the largest producers of waste and co2-emission
the volume and complexity of publicly available real estate data have been snowball

In [10]:
# Se presentan fórmulas retóricas de abstracts que aparecen en el corpus; no son conectores discursivos y se tratarán en una etapa distinta.

rhetorical_scaffolding = [
    "this study",
    "the study",
    "the development of",
    "the importance of",
    "the impact of",
    "in this paper",
    "the research",
    "the results",
    "the role of",
    "the paper",
    "focusing on",
    "the findings",
    "this study examines",
    "the author",
    "the application of",
    "to analyze",
    "the authors",
    "this article",
    "the effects of",
    "the integration of",
]

scaffolding_counts = pd.DataFrame(
    {
        "formula": rhetorical_scaffolding,
        "occurrences": [
            dataframe["normalized_text"].str.count(r"\b" + formula + r"\b").sum()
            for formula in rhetorical_scaffolding
        ],
    }
).query("occurrences > 0")

scaffolding_counts.sort_values("occurrences", ascending=False)

,formula,occurrences
0,this study,129
1,the study,80
7,the results,58
6,the research,41
9,the paper,41
11,the findings,38
17,this article,38
4,the impact of,35
2,the development of,31
16,the authors,28


In [11]:
# Se marcan las fórmulas retóricas para evidenciar que son unidades distintas antes de decidir su eliminación.

for formula in sorted(rhetorical_scaffolding, key=len, reverse=True):
    dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
        r"\b" + formula + r"\b",
        formula.replace(" ", "_"),
        regex=True,
    )

scaffolding_markers = [formula.replace(" ", "_") for formula in rhetorical_scaffolding]
scaffolding_pattern = "|".join(scaffolding_markers)
abstract_previews = dataframe["normalized_text"].str[:130]
marked_abstracts = abstract_previews[
    abstract_previews.str.contains(scaffolding_pattern)
].head(8)

for abstract in marked_abstracts:
    print(abstract)

this_study_examines the relationship between efficiency and the regulatory framework in malaysian fintech companies. it identifies
this review relates to the_development_of thin film (a few micrometers) marking means for identification of various objects and ar
in this_article by the editors of this special issue, the_authors discuss the changes being brought to real estate investment by n
this_study compares the performance of machine learning (ml) algorithms with traditional technical indicators in real estate, tech
key success factors of the malaysian fintech sector were explored among application-based platforms, focusing_on how trust, innova
the purpose of this_study is to prepare effective supply plans that account for the locational conditions and characteristics of d
purpose – the purpose of this_study is to address the significant challenge faced by startup online travel agencies (otas) in buil
this_article presents an overview on typical properties, technologies, and applicat

In [12]:
# Las fórmulas marcadas no distinguen los temas del corpus, por lo que se eliminan antes de tokenizar.

for marker in scaffolding_markers:
    dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
        r"\b" + marker + r"\b",
        " ",
        regex=True,
    )

dataframe["normalized_text"] = dataframe["normalized_text"].str.split().str.join(" ")

In [13]:
# Se separa cada abstract en tokens para que el texto se convierta en unidades procesables.

from nltk.tokenize import word_tokenize

dataframe["tokens"] = dataframe["normalized_text"].map(word_tokenize)

for tokens in dataframe["tokens"].head(8):
    print(" ".join(tokens)[:130])

the relationship between efficiency and the regulatory framework in malaysian fintech companies . it identifies three key themes f
the project started in 2018 following a use-case approach ( ie a descriptive and up-close exploratory examination of how a blockch
real estate has always been a significant investment sector , but it has a high barrier to entry . bitland project aims to use blo
global policy organizations have emphasized cash-crop production for poverty alleviation in the world 's semi-arid areas in recent
coffee is arguably the most popular drink in the world after water and it is estimated that global coffee consumption exceeds 2 bi
large-scale co2 emissions are believed to be a major contributor to global warming . co2 capture and storage ( ccs ) is the essent
real estate ( re ) is grappling with digital transformation . although property technology ( proptech ) start-ups are generating p
lychee ( litchi chinensis ) is one of the economic fruit in thailand with high nutr

In [14]:
# Se conservan tokens alfabéticos, incluidos los marcadores con guion bajo creados en las etapas anteriores.

dataframe["tokens"] = dataframe["tokens"].map(
    lambda tokens: [token for token in tokens if token.replace("_", "").isalpha()]
)

In [15]:
# Las stopwords frecuentes no diferencian los temas del corpus, por lo que se eliminan antes de formar cada documento procesado.

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopword_tokens = ENGLISH_STOP_WORDS

dataframe["tokens"] = dataframe["tokens"].map(
    lambda tokens: [token for token in tokens if token not in stopword_tokens]
)
dataframe["processed_text"] = dataframe["tokens"].map(" ".join)

print(" ".join(dataframe["tokens"].iloc[0])[:130])

relationship efficiency regulatory framework malaysian fintech companies identifies key themes interviews sectors payments proptec


In [16]:
# ¿Cómo se transforma la lista de tokens en una matriz documento–término apta para analizar el corpus?

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=2)
document_term_matrix = vectorizer.fit_transform(dataframe["processed_text"])
vocabulary = pd.DataFrame(
    {
        "token": vectorizer.get_feature_names_out(),
        "document_frequency": (document_term_matrix > 0).sum(axis=0).A1,
    }
)
document_term_matrix.shape, vocabulary.head()

((385, 3659),
           token  document_frequency
 0  abbreviation                   2
 1       ability                  15
 2          able                   9
 3    absorption                   4
 4      abstract                   3)

In [17]:
# Se guardan abstracts procesados, vocabulario y matriz para conservar una representación reproducible del corpus.

import json
from pathlib import Path
from scipy import sparse

submission_dir = Path("../submission")
dataframe.drop(columns="tokens").to_csv(
    submission_dir / "tokenized_abstracts.csv", index=False
)
vocabulary.to_csv(submission_dir / "vocabulary.csv", index=False)
sparse.save_npz(submission_dir / "document_term_matrix.npz", document_term_matrix)
with (submission_dir / "matrix_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "documents": document_term_matrix.shape[0],
            "terms": document_term_matrix.shape[1],
            "minimum_document_frequency": 2,
            "minimum_abstract_words": minimum_abstract_words,
            "excluded_abstracts": excluded_abstracts,
            "copyright_notices_removed": int(dataframe["has_copyright_notice"].sum()),
        },
        file,
        indent=2,
    )